# 11 run infercnvpy

**Purpose:** Run the five infercnvpy CNV strategies on the canonical notebook-10 inputs.

This repository version preserves the archived scientific parameters and adds
Windows/16-GB-RAM restart safety. Strategies run sequentially, and a completed
strategy is validated and skipped on rerun.

**Original executed analysis notebook:** `11_run_infercnvpy_CNV_HVG8000_r0_6_strategies_v2_fixed_cnv_columns.ipynb`


# 11 — infercnvpy CNV inference on HVG8000 r0.6 strategies

Historical executed environment: Scanpy 1.11.5, infercnvpy 0.6.1.

Historical infercnvpy settings are preserved: window_size=100, step=10,
dynamic_threshold=1.5, lfc_clip=3, chunksize=1000, n_jobs=2,
calculate_gene_values=False, followed by CNV PCA/neighbors/Leiden/UMAP.

Only one sampled strategy is loaded at a time. Each input is first copied to
NVMe scratch (`E:\glioma-cnv-scratch\main_analysis_11` by default), and the
large result H5AD is written on scratch before atomic publication to the
repository. Completed strategy outputs are validated and reused on rerun.

The canonical notebook-10 sampled cell identities may differ from the old
pre-renaming sampled objects because row order changed. Therefore structural
historical checkpoints and the final infercnvpy consensus are hard-checked;
archived floating-point thresholds are recorded as an audit rather than
required to be bit-identical.


In [1]:

# =========================
# Imports and global settings
# =========================

from pathlib import Path
import os
import gc
import time
import json
import shutil
import warnings
import h5py

import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import matplotlib.pyplot as plt

try:
    import infercnvpy as cnv
except Exception as e:
    raise ImportError(
        "infercnvpy could not be imported. Install it in the active environment first. "
        "Example: pip install infercnvpy"
    ) from e

warnings.filterwarnings("ignore")

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, facecolor="white")

# Keep BLAS/threading conservative for a 16 GB RAM machine.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("MALLOC_ARENA_MAX", "2")

print("scanpy:", sc.__version__)
print("infercnvpy:", getattr(cnv, "__version__", "unknown"))


HISTORICAL_SCANPY_VERSION = "1.11.5"
HISTORICAL_INFERCNVPY_VERSION = "0.6.1"

_scanpy_version = str(sc.__version__)
_infercnvpy_version = str(getattr(cnv, "__version__", "unknown"))

print("historical scanpy:", HISTORICAL_SCANPY_VERSION)
print("historical infercnvpy:", HISTORICAL_INFERCNVPY_VERSION)

if _scanpy_version != HISTORICAL_SCANPY_VERSION:
    print("[WARNING] Scanpy version differs from the archived executed run.")

if _infercnvpy_version != HISTORICAL_INFERCNVPY_VERSION:
    print("[WARNING] infercnvpy version differs from the archived executed run.")


scanpy: 1.11.5
infercnvpy: 0.6.1
historical scanpy: 1.11.5
historical infercnvpy: 0.6.1


C:\Users\AmirSilco\.conda\envs\glioma-cnv-atlas\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# =========================
# Paths and parameters
# =========================

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
RAW_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
RESULTS_DIR = PROJECT_DIR / "results"

CNV_INPUT_DIR = RAW_DIR / "integrated" / "CNV_inputs"
CNV_RESULT_DIR = RAW_DIR / "integrated" / "CNV_results" / "infercnvpy_HVG8000_r0_6"
CNV_FIGURE_DIR = FIGURES_DIR / "CNV_infercnvpy_HVG8000_r0_6"

for d in [METADATA_DIR, FIGURES_DIR, RESULTS_DIR, CNV_RESULT_DIR, CNV_FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

STRATEGY_MANIFEST_TSV = METADATA_DIR / "CNV_strategy_input_manifest_HVG8000_r0_6.tsv"
GENE_ORDER_TSV = METADATA_DIR / "CNV_gene_order_gencode_v49_matched_header_HVG8000_r0_6.tsv"

# Run order: main strategy first, robustness/review strategies after.
RUN_STRATEGIES = [
    "core_conservative",
    "core_plus_lymphoid",
    "extended_myeloid",
    "review_cycling",
    "review_ambiguous",
]

# infercnvpy parameters.
# Conservative memory settings for 16 GB RAM.
WINDOW_SIZE = 100
STEP = 10
DYNAMIC_THRESHOLD = 1.5
LFC_CLIP = 3
CHUNKSIZE = 1000
N_JOBS = 2
EXCLUDE_CHROMOSOMES = ("chrX", "chrY", "X", "Y", "chrM", "chrMT", "MT", "M")

# CNV graph / clustering parameters.
CNV_PCA_N_COMPS = 30
CNV_NEIGHBORS = 15
CNV_LEIDEN_RESOLUTION = 0.5
RANDOM_STATE = 42

# Main biological clustering key transferred from scVI.
CLUSTER_KEY = "leiden_scvi_main"
CNV_GROUP_KEY = "cnv_group"
REFERENCE_CAT = "reference"
QUERY_CAT = "query"

OVERWRITE = True

EXPECTED_N_GENES = 15163
EXPECTED_N_CNV_WINDOWS = 1256

EXPECTED_STRATEGY_CELLS = {
    "core_conservative": 32640,
    "core_plus_lymphoid": 38807,
    "extended_myeloid": 45000,
    "review_cycling": 14640,
    "review_ambiguous": 19443,
}

EXPECTED_STRATEGY_CLUSTER_ROWS = {
    "core_conservative": 6,
    "core_plus_lymphoid": 8,
    "extended_myeloid": 9,
    "review_cycling": 3,
    "review_ambiguous": 4,
}

def _default_main11_scratch():
    configured = os.environ.get("GLIOMA_MAIN11_SCRATCH_ROOT")
    if configured:
        return Path(configured).expanduser().resolve()
    if os.name == "nt" and Path("E:/").exists():
        return Path(r"E:\glioma-cnv-scratch\main_analysis_11")
    return (PROJECT_DIR / ".scratch" / "main_analysis_11").resolve()

SCRATCH_ROOT = _default_main11_scratch()
SCRATCH_INPUT_DIR = SCRATCH_ROOT / "inputs"
SCRATCH_RESULT_DIR = SCRATCH_ROOT / "results"

for d in [SCRATCH_ROOT, SCRATCH_INPUT_DIR, SCRATCH_RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Strategy manifest:", STRATEGY_MANIFEST_TSV)
print("CNV result dir:", CNV_RESULT_DIR)
print("CNV figure dir:", CNV_FIGURE_DIR)

print("Scratch root:", SCRATCH_ROOT)


Strategy manifest: G:\Repository\glioma-cnv-single-cell-atlas\metadata\CNV_strategy_input_manifest_HVG8000_r0_6.tsv
CNV result dir: G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_results\infercnvpy_HVG8000_r0_6
CNV figure dir: G:\Repository\glioma-cnv-single-cell-atlas\figures\CNV_infercnvpy_HVG8000_r0_6
Scratch root: E:\glioma-cnv-scratch\main_analysis_11


In [3]:

# =========================
# Utility functions
# =========================

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")


def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")


def matrix_values_sample(X, n_rows=1000, n_cols=1000):
    xr = X[:min(X.shape[0], n_rows), :min(X.shape[1], n_cols)]
    if sp.issparse(xr):
        vals = xr.data
    else:
        vals = np.asarray(xr).ravel()
    vals = vals[np.isfinite(vals)]
    return vals


def matrix_is_integer_like(X):
    vals = matrix_values_sample(X)
    if vals.size == 0:
        return True
    return bool(np.allclose(vals, np.round(vals)))


def has_negative_values(X):
    vals = matrix_values_sample(X)
    if vals.size == 0:
        return False
    return bool(np.any(vals < 0))


def natural_chrom_sort_key(chrom):
    c = str(chrom).replace("chr", "")
    if c.isdigit():
        return (0, int(c))
    if c == "X":
        return (1, 23)
    if c == "Y":
        return (1, 24)
    if c in ["M", "MT"]:
        return (2, 25)
    return (3, c)


def standardize_gene_position_columns(adata):
    """Historical gene-position standardization with no unnecessary matrix copy."""
    var = adata.var

    chrom_candidates = [
        "chromosome", "chrom", "chr", "seqnames",
        "gene_chr", "gene_chromosome",
        "cnv_chr", "cnv_chrom", "cnv_chromosome",
    ]
    start_candidates = [
        "start", "gene_start", "tx_start", "begin",
        "cnv_start", "cnv_gene_start",
    ]
    end_candidates = [
        "end", "gene_end", "tx_end", "stop",
        "cnv_end", "cnv_gene_end",
    ]

    def find_col(candidates):
        for column in candidates:
            if column in var.columns:
                return column
        return None

    chrom_col = find_col(chrom_candidates)
    start_col = find_col(start_candidates)
    end_col = find_col(end_candidates)

    if chrom_col is None or start_col is None or end_col is None:
        raise ValueError(
            "Gene position columns not found in adata.var. "
            f"Available var columns: {list(var.columns)}"
        )

    def normalize_chr(value):
        value = str(value)
        if value.startswith("chr"):
            return value
        if value in [
            "1","2","3","4","5","6","7","8","9","10","11","12",
            "13","14","15","16","17","18","19","20","21","22",
            "X","Y","M","MT",
        ]:
            return "chr" + value
        return value

    adata.var["chromosome"] = adata.var[chrom_col].astype(str).map(normalize_chr)
    adata.var["start"] = pd.to_numeric(adata.var[start_col], errors="coerce")
    adata.var["end"] = pd.to_numeric(adata.var[end_col], errors="coerce")

    valid = (
        adata.var["chromosome"].notna()
        & adata.var["start"].notna()
        & adata.var["end"].notna()
    )
    if int(valid.sum()) != adata.n_vars:
        raise RuntimeError(
            "Notebook-10 gene-ordered input contains genes without genomic positions."
        )

    sort_df = pd.DataFrame(
        {
            "gene": adata.var_names.astype(str),
            "chromosome": adata.var["chromosome"].astype(str).values,
            "start": adata.var["start"].astype(float).values,
            "end": adata.var["end"].astype(float).values,
        },
        index=adata.var_names,
    )
    sort_df["chrom_sort"] = sort_df["chromosome"].map(natural_chrom_sort_key)
    sorted_names = sort_df.sort_values(
        ["chrom_sort", "start", "end", "gene"]
    ).index

    if adata.var_names.equals(sorted_names):
        print("[GENE ORDER] already historical-compatible; no matrix copy.")
        return adata

    print("[GENE ORDER] applying historical column order; one sparse copy required.")
    return adata[:, sorted_names].copy()


def atomic_copy(source, destination):
    source = Path(source)
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".publishing")
    if temporary.exists():
        temporary.unlink()
    shutil.copyfile(source, temporary)
    if temporary.stat().st_size != source.stat().st_size:
        raise IOError(f"Copy-size mismatch: {source} -> {destination}")
    os.replace(temporary, destination)


def normalize_cnv_umap_storage_key(path, verbose=False):
    """
    Compatibility repair for infercnvpy/Scanpy UMAP key naming.

    Historical code requests key_added="cnv_umap". Some combinations may
    serialize the embedding as obsm["X_cnv_umap"]. If so, create an HDF5
    hard-link named "cnv_umap" to the same stored array. This duplicates no
    numeric data and changes no CNV result.
    """
    path = Path(path)
    if not path.exists():
        return False

    try:
        with h5py.File(path, "r+") as handle:
            if "obsm" not in handle:
                return False

            obsm = handle["obsm"]

            if "cnv_umap" in obsm:
                return True

            if "X_cnv_umap" in obsm:
                obsm["cnv_umap"] = obsm["X_cnv_umap"]
                handle.flush()
                if verbose:
                    print(
                        "[COMPATIBILITY REPAIR] created /obsm/cnv_umap "
                        "as a hard-link to /obsm/X_cnv_umap"
                    )
                return True

            return False

    except OSError:
        # Read-only/non-writable path: validation below will report the actual key.
        return False


def validate_result_h5ad(path, expected_cells, verbose=False):
    path = Path(path)

    if not path.exists():
        if verbose:
            print("[VALIDATION FAIL] missing file:", path)
        return False

    # First repair only the known zero-copy UMAP-key compatibility case.
    normalize_cnv_umap_storage_key(path, verbose=verbose)

    check = sc.read_h5ad(path, backed="r")
    try:
        problems = []

        if check.shape != (int(expected_cells), EXPECTED_N_GENES):
            problems.append(
                f"shape={check.shape}, expected={(int(expected_cells), EXPECTED_N_GENES)}"
            )

        obsm_keys = list(check.obsm.keys())
        obs_columns = set(check.obs.columns)

        if "X_cnv" not in check.obsm:
            problems.append(f"X_cnv missing; obsm keys={obsm_keys}")
        else:
            observed_x_cnv_shape = tuple(check.obsm["X_cnv"].shape)
            expected_x_cnv_shape = (
                int(expected_cells),
                EXPECTED_N_CNV_WINDOWS,
            )
            if observed_x_cnv_shape != expected_x_cnv_shape:
                problems.append(
                    f"X_cnv shape={observed_x_cnv_shape}, "
                    f"expected={expected_x_cnv_shape}"
                )

        required_obs = [
            CNV_GROUP_KEY,
            CLUSTER_KEY,
            "cnv_abs_mean_score",
            "cnv_leiden",
        ]
        missing_obs = [
            column
            for column in required_obs
            if column not in obs_columns
        ]
        if missing_obs:
            problems.append(f"missing obs columns={missing_obs}")

        if "cnv_umap" not in check.obsm:
            problems.append(
                "cnv_umap missing after compatibility repair; "
                f"obsm keys={obsm_keys}"
            )

        if problems:
            if verbose:
                print("[RESULT VALIDATION FAILED]")
                for problem in problems:
                    print(" -", problem)
            return False

        if verbose:
            print(
                "[RESULT VALIDATED]",
                path,
                "| shape=", check.shape,
                "| X_cnv=", tuple(check.obsm["X_cnv"].shape),
                "| obsm=", list(check.obsm.keys()),
            )
        return True

    finally:
        check.file.close()


def strategy_outputs_valid(strategy, expected_cells):
    prefix = f"infercnvpy_{strategy}_HVG8000_r0_6"
    result_h5ad = CNV_RESULT_DIR / f"{prefix}_result.h5ad"
    cell_scores_tsv = METADATA_DIR / f"{prefix}_cell_scores.tsv"
    cluster_summary_tsv = METADATA_DIR / f"{prefix}_cluster_summary.tsv"
    cnv_leiden_summary_tsv = METADATA_DIR / f"{prefix}_cnv_leiden_summary.tsv"
    run_summary_tsv = METADATA_DIR / f"{prefix}_run_summary.tsv"

    if not all(path.exists() for path in [
        result_h5ad, cell_scores_tsv, cluster_summary_tsv,
        cnv_leiden_summary_tsv, run_summary_tsv,
    ]):
        return False

    if not validate_result_h5ad(result_h5ad, expected_cells):
        return False

    try:
        run_summary = pd.read_csv(run_summary_tsv, sep="	")
        cluster_summary = pd.read_csv(cluster_summary_tsv, sep="	")
        cell_scores = pd.read_csv(cell_scores_tsv, sep="	", nrows=3)
        cnv_leiden_summary = pd.read_csv(cnv_leiden_summary_tsv, sep="	")
    except Exception:
        return False

    if run_summary.shape != (1, 18):
        return False
    if int(run_summary.loc[0, "n_cells"]) != int(expected_cells):
        return False
    if int(run_summary.loc[0, "n_genes"]) != EXPECTED_N_GENES:
        return False
    if int(run_summary.loc[0, "n_cnv_windows"]) != EXPECTED_N_CNV_WINDOWS:
        return False
    if cluster_summary.shape != (EXPECTED_STRATEGY_CLUSTER_ROWS[strategy], 30):
        return False
    if cell_scores.shape[1] != 13:
        return False
    if cnv_leiden_summary.empty or cnv_leiden_summary.shape[1] != 18:
        return False

    return True


def existing_run_record(strategy):
    prefix = f"infercnvpy_{strategy}_HVG8000_r0_6"
    return {
        "strategy": strategy,
        "run_summary_tsv": str(METADATA_DIR / f"{prefix}_run_summary.tsv"),
        "cluster_summary_tsv": str(METADATA_DIR / f"{prefix}_cluster_summary.tsv"),
        "cell_scores_tsv": str(METADATA_DIR / f"{prefix}_cell_scores.tsv"),
        "cnv_leiden_summary_tsv": str(METADATA_DIR / f"{prefix}_cnv_leiden_summary.tsv"),
        "result_h5ad": str(CNV_RESULT_DIR / f"{prefix}_result.h5ad"),
    }


def compute_abs_cnv_score(adata, cnv_key="X_cnv"):
    if cnv_key not in adata.obsm:
        raise KeyError(f"{cnv_key} not found in adata.obsm")
    X = adata.obsm[cnv_key]
    if sp.issparse(X):
        score = np.asarray(np.abs(X).mean(axis=1)).ravel()
    else:
        score = np.mean(np.abs(np.asarray(X)), axis=1)
    return score


def top_fraction(series):
    counts = series.astype(str).value_counts(dropna=False)
    if counts.empty:
        return pd.Series({"top": "NA", "top_n": 0, "top_fraction": np.nan, "n_unique": 0})
    return pd.Series({
        "top": counts.index[0],
        "top_n": int(counts.iloc[0]),
        "top_fraction": float(counts.iloc[0] / counts.sum()),
        "n_unique": int(counts.shape[0]),
    })


def robust_ref_thresholds(ref_scores):
    ref_scores = np.asarray(ref_scores, dtype=float)
    ref_scores = ref_scores[np.isfinite(ref_scores)]
    med = float(np.median(ref_scores))
    mad = float(np.median(np.abs(ref_scores - med)))
    if mad == 0:
        mad = float(np.std(ref_scores))
    if mad == 0:
        mad = 1e-9
    q95 = float(np.quantile(ref_scores, 0.95))
    q99 = float(np.quantile(ref_scores, 0.99))
    moderate = max(q95, med + 2 * 1.4826 * mad)
    high = max(q99, med + 3 * 1.4826 * mad)
    return {
        "reference_median": med,
        "reference_mad": mad,
        "reference_q95": q95,
        "reference_q99": q99,
        "moderate_threshold": float(moderate),
        "high_threshold": float(high),
    }


def call_cluster(mean_score, median_score, thresholds):
    score = max(float(mean_score), float(median_score))
    if score >= thresholds["high_threshold"]:
        return "CNV_high_candidate"
    if score >= thresholds["moderate_threshold"]:
        return "CNV_intermediate_candidate"
    return "CNV_low_or_reference_like"


In [4]:

# =========================
# Read strategy manifest and validate input files
# =========================

if not STRATEGY_MANIFEST_TSV.exists():
    raise FileNotFoundError(STRATEGY_MANIFEST_TSV)

manifest = pd.read_csv(STRATEGY_MANIFEST_TSV, sep="\t")
manifest = manifest[manifest["strategy"].isin(RUN_STRATEGIES)].copy()
manifest["strategy"] = pd.Categorical(manifest["strategy"], categories=RUN_STRATEGIES, ordered=True)
manifest = manifest.sort_values("strategy").reset_index(drop=True)

rows = []
for _, r in manifest.iterrows():
    h5 = Path(r["sampled_h5ad"])
    rows.append({
        "strategy": r["strategy"],
        "sampled_h5ad": str(h5),
        "exists": h5.exists(),
        "size_mb": round(h5.stat().st_size / 1024**2, 3) if h5.exists() else np.nan,
        "reference_clusters": r["reference_clusters"],
        "query_clusters": r["query_clusters"],
        "n_sampled_reference_cells": r["n_sampled_reference_cells"],
        "n_sampled_query_cells": r["n_sampled_query_cells"],
        "n_sampled_total_cells": r["n_sampled_total_cells"],
    })

readiness = pd.DataFrame(rows)
READINESS_TSV = METADATA_DIR / "infercnvpy_run_readiness_HVG8000_r0_6.tsv"
write_tsv_replace(readiness, READINESS_TSV)

display(readiness)

if not readiness["exists"].all():
    missing = readiness.loc[~readiness["exists"], "sampled_h5ad"].tolist()
    raise FileNotFoundError(f"Missing strategy h5ad files: {missing}")


if readiness.shape != (5, 9):
    raise AssertionError(f"Unexpected readiness shape: {readiness.shape}")

for _, row in readiness.iterrows():
    strategy = str(row["strategy"])
    if int(row["n_sampled_total_cells"]) != EXPECTED_STRATEGY_CELLS[strategy]:
        raise AssertionError(
            f"{strategy} sampled-cell count changed: "
            f"{row['n_sampled_total_cells']} vs {EXPECTED_STRATEGY_CELLS[strategy]}"
        )

    check = sc.read_h5ad(Path(row["sampled_h5ad"]), backed="r")
    try:
        if check.shape != (EXPECTED_STRATEGY_CELLS[strategy], EXPECTED_N_GENES):
            raise AssertionError(
                f"{strategy} input shape changed: {check.shape}"
            )
        if CNV_GROUP_KEY not in check.obs.columns or CLUSTER_KEY not in check.obs.columns:
            raise KeyError(f"{strategy} is missing CNV/cluster metadata.")
        if not {"cnv_chr", "cnv_start", "cnv_end"}.issubset(check.var.columns):
            raise KeyError(f"{strategy} is missing notebook-10 genomic coordinates.")
        group_counts = check.obs[CNV_GROUP_KEY].astype(str).value_counts()
        if int(group_counts.get("reference", 0)) != int(row["n_sampled_reference_cells"]):
            raise AssertionError(f"{strategy} reference-cell count changed.")
        if int(group_counts.get("query", 0)) != int(row["n_sampled_query_cells"]):
            raise AssertionError(f"{strategy} query-cell count changed.")
    finally:
        check.file.close()

print("[VALIDATED] all five notebook-10 strategy inputs")


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_run_readiness_HVG8000_r0_6.tsv shape=(5, 9)


,strategy,sampled_h5ad,exists,size_mb,reference_clusters,query_clusters,n_sampled_reference_cells,n_sampled_query_cells,n_sampled_total_cells
0,core_conservative,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,164.129,"8,14","0,4,7,11",7640,25000,32640
1,core_plus_lymphoid,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,186.050,"8,14,5,15","0,4,7,11",13807,25000,38807
2,extended_myeloid,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,223.359,"8,14,1,3,6","0,4,7,11",20000,25000,45000
3,review_cycling,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,83.434,"8,14",2,7640,7000,14640
4,review_ambiguous,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,84.256,"8,14","9,12",7640,11803,19443


[VALIDATED] all five notebook-10 strategy inputs


In [5]:

# =========================
# Main per-strategy runner
# =========================

def run_one_infercnvpy_strategy(strategy_row):
    strategy = str(strategy_row["strategy"])
    input_h5ad = Path(strategy_row["sampled_h5ad"])

    print("\n" + "=" * 100)
    print(f"[START] {strategy}")
    print(f"Input: {input_h5ad}")
    t0 = time.time()

    out_prefix = f"infercnvpy_{strategy}_HVG8000_r0_6"
    result_h5ad = CNV_RESULT_DIR / f"{out_prefix}_result.h5ad"
    cell_scores_tsv = METADATA_DIR / f"{out_prefix}_cell_scores.tsv"
    cluster_summary_tsv = METADATA_DIR / f"{out_prefix}_cluster_summary.tsv"
    cnv_leiden_summary_tsv = METADATA_DIR / f"{out_prefix}_cnv_leiden_summary.tsv"
    run_summary_tsv = METADATA_DIR / f"{out_prefix}_run_summary.tsv"

    expected_cells = EXPECTED_STRATEGY_CELLS[strategy]

    if strategy_outputs_valid(strategy, expected_cells):
        print("[SKIP] validated published outputs already exist.")
        return existing_run_record(strategy)

    # ------------------------------------------------------------
    # Resume a completed scratch result from an interrupted publication.
    # This specifically preserves expensive infercnvpy work that already
    # finished before a post-write validation/publishing failure.
    # ------------------------------------------------------------
    scratch_result_h5ad = (
        SCRATCH_RESULT_DIR
        / f"infercnvpy_{strategy}_HVG8000_r0_6_result.h5ad"
    )

    prefix = f"infercnvpy_{strategy}_HVG8000_r0_6"
    result_h5ad = CNV_RESULT_DIR / f"{prefix}_result.h5ad"
    cell_scores_tsv = METADATA_DIR / f"{prefix}_cell_scores.tsv"
    cluster_summary_tsv = METADATA_DIR / f"{prefix}_cluster_summary.tsv"
    cnv_leiden_summary_tsv = METADATA_DIR / f"{prefix}_cnv_leiden_summary.tsv"
    run_summary_tsv = METADATA_DIR / f"{prefix}_run_summary.tsv"

    if scratch_result_h5ad.exists():
        print(
            "[FOUND SCRATCH RESULT]",
            scratch_result_h5ad,
        )

        scratch_ok = validate_result_h5ad(
            scratch_result_h5ad,
            expected_cells,
            verbose=True,
        )

        supporting_tsvs_ok = all(
            path.exists()
            for path in [
                cell_scores_tsv,
                cluster_summary_tsv,
                cnv_leiden_summary_tsv,
            ]
        )

        if scratch_ok and supporting_tsvs_ok:
            print(
                "[RESUME] completed scratch infercnvpy result is valid; "
                "publishing without recomputation."
            )

            atomic_copy(
                scratch_result_h5ad,
                result_h5ad,
            )

            # Reconstruct the one-row run summary that was not reached when
            # the previous run stopped at post-write validation.
            scratch_check = sc.read_h5ad(
                scratch_result_h5ad,
                backed="r",
            )
            try:
                run_info = scratch_check.uns.get(
                    "infercnvpy_HVG8000_r0_6_run_info",
                    {},
                )

                thresholds = dict(
                    run_info.get(
                        "thresholds",
                        {},
                    )
                )

                preprocessing = str(
                    run_info.get(
                        "preprocessing",
                        "normalize_total_1e4_log1p_from_raw_counts_X",
                    )
                )

                n_cnv_windows = int(
                    scratch_check.obsm["X_cnv"].shape[1]
                )

            finally:
                scratch_check.file.close()

            # Preserve the historical 18-column schema. Elapsed time cannot
            # be reconstructed after an interrupted publication, so NaN is
            # explicit and scientifically irrelevant.
            run_summary = pd.DataFrame(
                [
                    {
                        "strategy": strategy,
                        "input_h5ad": str(input_h5ad),
                        "result_h5ad": str(result_h5ad),
                        "cell_scores_tsv": str(cell_scores_tsv),
                        "cluster_summary_tsv": str(cluster_summary_tsv),
                        "cnv_leiden_summary_tsv": str(cnv_leiden_summary_tsv),
                        "n_cells": int(expected_cells),
                        "n_genes": EXPECTED_N_GENES,
                        "n_cnv_windows": n_cnv_windows,
                        "preprocessing": preprocessing,
                        "elapsed_minutes": np.nan,
                        "figures": "",
                        **thresholds,
                    }
                ]
            )

            if run_summary.shape != (1, 18):
                raise RuntimeError(
                    "Recovered run-summary schema is not historical-compatible: "
                    f"{run_summary.shape}"
                )

            write_tsv_replace(
                run_summary,
                run_summary_tsv,
            )

            if not strategy_outputs_valid(
                strategy,
                expected_cells,
            ):
                raise RuntimeError(
                    f"{strategy} recovered/published outputs failed validation."
                )

            print(
                "[RECOVERED WITHOUT RECOMPUTATION]",
                strategy,
            )

            return existing_run_record(
                strategy
            )

        print(
            "[SCRATCH RESULT NOT RECOVERABLE] "
            "The notebook will recompute this strategy."
        )

    scratch_input = SCRATCH_INPUT_DIR / input_h5ad.name
    if (
        not scratch_input.exists()
        or scratch_input.stat().st_size != input_h5ad.stat().st_size
    ):
        if scratch_input.exists():
            scratch_input.unlink()
        print("[COPY INPUT TO NVMe]", input_h5ad, "->", scratch_input)
        shutil.copyfile(input_h5ad, scratch_input)

    if scratch_input.stat().st_size != input_h5ad.stat().st_size:
        raise IOError("Scratch input copy-size mismatch.")

    adata = sc.read_h5ad(scratch_input)
    print(adata)

    if CNV_GROUP_KEY not in adata.obs.columns:
        raise KeyError(f"{CNV_GROUP_KEY} not found in adata.obs. Available: {list(adata.obs.columns)}")
    if CLUSTER_KEY not in adata.obs.columns:
        raise KeyError(f"{CLUSTER_KEY} not found in adata.obs. Available: {list(adata.obs.columns)}")

    adata.obs[CNV_GROUP_KEY] = adata.obs[CNV_GROUP_KEY].astype(str)
    adata.obs[CLUSTER_KEY] = adata.obs[CLUSTER_KEY].astype(str)

    # Ensure gene positions are present and sorted.
    adata = standardize_gene_position_columns(adata)

    if adata.shape != (expected_cells, EXPECTED_N_GENES):
        raise AssertionError(
            f"{strategy} shape changed during gene-position standardization: {adata.shape}"
        )

    # Record input count status before log-normalization.
    input_integer_like = matrix_is_integer_like(adata.X)
    input_has_negative = has_negative_values(adata.X)
    if input_has_negative:
        raise ValueError(f"Negative values found in X for {strategy}. CNV input should not contain negative counts/log values.")

    # infercnvpy expects normalized/log-transformed expression.
    # The sampled input X is raw counts from the previous notebook, so normalize/log1p if it looks count-like.
    preprocessing = "unknown"
    if not input_integer_like:
        raise ValueError(
            f"{strategy} X no longer looks like raw counts; "
            "the archived preprocessing cannot be reproduced safely."
        )

    print("[PREPROCESS] X looks like raw counts; applying normalize_total + log1p.")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    preprocessing = "normalize_total_1e4_log1p_from_raw_counts_X"

    # Run infercnvpy. Do not calculate per-gene values to save memory.
    cnv.tl.infercnv(
        adata,
        reference_key=CNV_GROUP_KEY,
        reference_cat=REFERENCE_CAT,
        lfc_clip=LFC_CLIP,
        window_size=WINDOW_SIZE,
        step=STEP,
        dynamic_threshold=DYNAMIC_THRESHOLD,
        exclude_chromosomes=EXCLUDE_CHROMOSOMES,
        chunksize=CHUNKSIZE,
        n_jobs=N_JOBS,
        layer=None,
        key_added="cnv",
        calculate_gene_values=False,
    )

    if "X_cnv" not in adata.obsm:
        raise KeyError("infercnvpy did not create adata.obsm['X_cnv']")

    if tuple(adata.obsm["X_cnv"].shape) != (expected_cells, EXPECTED_N_CNV_WINDOWS):
        raise AssertionError(
            f"{strategy} X_cnv shape changed: {adata.obsm['X_cnv'].shape}"
        )

    # Manual per-cell CNV burden. This is independent of cnv clustering.
    adata.obs["cnv_abs_mean_score"] = compute_abs_cnv_score(adata, "X_cnv")

    # CNV embedding / clustering. These are helpful diagnostics, not final labels.
    try:
        cnv.tl.pca(
            adata,
            use_rep="cnv",
            key_added="cnv_pca",
            n_comps=CNV_PCA_N_COMPS,
            svd_solver="arpack",
            zero_center=False,
        )
        cnv.pp.neighbors(
            adata,
            use_rep="cnv_pca",
            key_added="cnv_neighbors",
            n_neighbors=CNV_NEIGHBORS,
            random_state=RANDOM_STATE,
        )
        cnv.tl.leiden(
            adata,
            neighbors_key="cnv_neighbors",
            key_added="cnv_leiden",
            resolution=CNV_LEIDEN_RESOLUTION,
            random_state=RANDOM_STATE,
        )
        cnv.tl.umap(
            adata,
            neighbors_key="cnv_neighbors",
            key_added="cnv_umap",
            random_state=RANDOM_STATE,
        )

        # infercnvpy/Scanpy compatibility: preserve the historical requested
        # key "cnv_umap" even if the installed stack returns "X_cnv_umap".
        if "cnv_umap" not in adata.obsm and "X_cnv_umap" in adata.obsm:
            adata.obsm["cnv_umap"] = adata.obsm["X_cnv_umap"]
            print(
                "[COMPATIBILITY] aliased obsm['X_cnv_umap'] "
                "to obsm['cnv_umap']"
            )

        if "cnv_umap" not in adata.obsm:
            raise RuntimeError(
                "CNV UMAP completed without the expected cnv_umap embedding. "
                f"Available obsm keys: {list(adata.obsm.keys())}"
            )

        cnv.tl.cnv_score(
            adata,
            groupby="cnv_leiden",
            use_rep="cnv",
            key_added="cnv_cluster_score",
        )
    except Exception as e:
        print("[WARNING] CNV PCA/neighbors/leiden/UMAP failed:", repr(e))

    ref_scores = adata.obs.loc[adata.obs[CNV_GROUP_KEY] == REFERENCE_CAT, "cnv_abs_mean_score"].astype(float).values
    thresholds = robust_ref_thresholds(ref_scores)

    # Cluster-level summary based on the biological scVI cluster key.
    summary_rows = []
    meta_cols = [
        "core_dataset", "condition", "patient_id", "sample_id",
        "annotation_marker_based_HVG8000_r0_6",
        "lineage_marker_based_HVG8000_r0_6",
        "annotation_confidence_HVG8000_r0_6",
        "cnv_role_HVG8000_r0_6",
        "bias_status_HVG8000_r0_6",
    ]
    present_meta_cols = [c for c in meta_cols if c in adata.obs.columns]

    for cl, df in adata.obs.groupby(CLUSTER_KEY, observed=True):
        scores = df["cnv_abs_mean_score"].astype(float).values
        groups = df[CNV_GROUP_KEY].astype(str).value_counts()
        main_group = groups.index[0] if len(groups) else "NA"
        row = {
            "strategy": strategy,
            "cluster": str(cl),
            "n_cells": int(df.shape[0]),
            "cnv_group_majority": main_group,
            "cnv_group_majority_fraction": float(groups.iloc[0] / groups.sum()) if len(groups) else np.nan,
            "mean_cnv_abs_score": float(np.mean(scores)),
            "median_cnv_abs_score": float(np.median(scores)),
            "q75_cnv_abs_score": float(np.quantile(scores, 0.75)),
            "q90_cnv_abs_score": float(np.quantile(scores, 0.90)),
            "reference_median": thresholds["reference_median"],
            "reference_q95": thresholds["reference_q95"],
            "reference_q99": thresholds["reference_q99"],
            "moderate_threshold": thresholds["moderate_threshold"],
            "high_threshold": thresholds["high_threshold"],
            "cnv_call": call_cluster(np.mean(scores), np.median(scores), thresholds),
        }
        # Add robust z-like effect based on reference median/MAD.
        row["robust_effect_vs_reference_median"] = float((row["median_cnv_abs_score"] - thresholds["reference_median"]) / (1.4826 * thresholds["reference_mad"] + 1e-9))

        for col in present_meta_cols:
            vc = df[col].astype(str).value_counts(dropna=False)
            row[f"top_{col}"] = vc.index[0] if len(vc) else "NA"
            row[f"top_{col}_fraction"] = float(vc.iloc[0] / vc.sum()) if len(vc) else np.nan

        summary_rows.append(row)

    cluster_summary = pd.DataFrame(summary_rows)
    if cluster_summary.shape != (EXPECTED_STRATEGY_CLUSTER_ROWS[strategy], 30):
        raise AssertionError(
            f"{strategy} cluster-summary shape changed: {cluster_summary.shape}"
        )
    write_tsv_replace(cluster_summary, cluster_summary_tsv)

    # CNV-leiden summary if available.
    if "cnv_leiden" in adata.obs.columns:
        cnv_leiden_rows = []
        for cl, df in adata.obs.groupby("cnv_leiden", observed=True):
            scores = df["cnv_abs_mean_score"].astype(float).values
            row = {
                "strategy": strategy,
                "cnv_leiden": str(cl),
                "n_cells": int(df.shape[0]),
                "mean_cnv_abs_score": float(np.mean(scores)),
                "median_cnv_abs_score": float(np.median(scores)),
                "q90_cnv_abs_score": float(np.quantile(scores, 0.90)),
            }
            for col in [CNV_GROUP_KEY, CLUSTER_KEY, "core_dataset", "condition", "patient_id", "sample_id"]:
                if col in df.columns:
                    vc = df[col].astype(str).value_counts(dropna=False)
                    row[f"top_{col}"] = vc.index[0] if len(vc) else "NA"
                    row[f"top_{col}_fraction"] = float(vc.iloc[0] / vc.sum()) if len(vc) else np.nan
            cnv_leiden_rows.append(row)
        cnv_leiden_summary = pd.DataFrame(cnv_leiden_rows)
    else:
        cnv_leiden_summary = pd.DataFrame()
    write_tsv_replace(cnv_leiden_summary, cnv_leiden_summary_tsv)

    # Per-cell scores, limited to essential columns.
    cell_cols = [
        CNV_GROUP_KEY, CLUSTER_KEY, "cnv_abs_mean_score", "core_dataset", "condition", "patient_id", "sample_id",
        "annotation_marker_based_HVG8000_r0_6", "lineage_marker_based_HVG8000_r0_6",
    ]
    if "cnv_leiden" in adata.obs.columns:
        cell_cols.append("cnv_leiden")
    if "cnv_cluster_score" in adata.obs.columns:
        cell_cols.append("cnv_cluster_score")
    cell_cols = [c for c in cell_cols if c in adata.obs.columns]
    cell_scores = adata.obs[cell_cols].copy()
    cell_scores.insert(0, "cell_id", adata.obs_names.astype(str))
    cell_scores.insert(0, "strategy", strategy)
    if cell_scores.shape != (expected_cells, 13):
        raise AssertionError(
            f"{strategy} cell-score shape changed: {cell_scores.shape}"
        )
    write_tsv_replace(cell_scores, cell_scores_tsv)

    # Figures.
    fig_paths = []
    try:
        if "cnv_umap" in adata.obsm:
            for color in [CNV_GROUP_KEY, CLUSTER_KEY, "cnv_abs_mean_score", "cnv_leiden"]:
                if color in adata.obs.columns:
                    fig = sc.pl.embedding(
                        adata,
                        basis="cnv_umap",
                        color=color,
                        show=False,
                        return_fig=True,
                        title=f"{strategy} | CNV UMAP | {color}",
                    )
                    fpath = CNV_FIGURE_DIR / f"{out_prefix}_cnv_umap_{color}.png"
                    fig.savefig(fpath, dpi=160, bbox_inches="tight")
                    plt.close(fig)
                    fig_paths.append(str(fpath))
    except Exception as e:
        print("[WARNING] CNV UMAP plotting failed:", repr(e))

    try:
        for groupby in [CNV_GROUP_KEY, CLUSTER_KEY]:
            if groupby in adata.obs.columns:
                cnv.pl.chromosome_heatmap_summary(
                    adata,
                    groupby=groupby,
                    use_rep="cnv",
                    figsize=(18, 8),
                    show=False,
                )
                fpath = CNV_FIGURE_DIR / f"{out_prefix}_chromosome_heatmap_summary_by_{groupby}.png"
                plt.savefig(fpath, dpi=160, bbox_inches="tight")
                plt.close("all")
                fig_paths.append(str(fpath))
    except Exception as e:
        print("[WARNING] chromosome heatmap summary failed:", repr(e))

    # Store run metadata in uns in a JSON-serializable way.
    adata.uns["infercnvpy_HVG8000_r0_6_run_info"] = {
        "strategy": strategy,
        "input_h5ad": str(input_h5ad),
        "preprocessing": preprocessing,
        "reference_key": CNV_GROUP_KEY,
        "reference_cat": REFERENCE_CAT,
        "window_size": WINDOW_SIZE,
        "step": STEP,
        "dynamic_threshold": DYNAMIC_THRESHOLD,
        "lfc_clip": LFC_CLIP,
        "chunksize": CHUNKSIZE,
        "n_jobs": N_JOBS,
        "exclude_chromosomes": list(EXCLUDE_CHROMOSOMES),
        "thresholds": thresholds,
    }

    scratch_result_h5ad = SCRATCH_RESULT_DIR / result_h5ad.name
    if scratch_result_h5ad.exists():
        scratch_result_h5ad.unlink()

    adata.write_h5ad(scratch_result_h5ad)

    if not validate_result_h5ad(scratch_result_h5ad, expected_cells, verbose=True):
        raise RuntimeError(f"{strategy} scratch result failed validation.")

    atomic_copy(scratch_result_h5ad, result_h5ad)

    if not validate_result_h5ad(result_h5ad, expected_cells, verbose=True):
        raise RuntimeError(f"{strategy} published result failed validation.")

    elapsed_min = (time.time() - t0) / 60
    run_summary = pd.DataFrame([{
        "strategy": strategy,
        "input_h5ad": str(input_h5ad),
        "result_h5ad": str(result_h5ad),
        "cell_scores_tsv": str(cell_scores_tsv),
        "cluster_summary_tsv": str(cluster_summary_tsv),
        "cnv_leiden_summary_tsv": str(cnv_leiden_summary_tsv),
        "n_cells": int(adata.n_obs),
        "n_genes": int(adata.n_vars),
        "n_cnv_windows": int(adata.obsm["X_cnv"].shape[1]) if "X_cnv" in adata.obsm else np.nan,
        "preprocessing": preprocessing,
        "elapsed_minutes": round(elapsed_min, 3),
        "figures": ";".join(fig_paths),
        **thresholds,
    }])
    write_tsv_replace(run_summary, run_summary_tsv)

    if not strategy_outputs_valid(strategy, expected_cells):
        raise RuntimeError(
            f"{strategy} published outputs failed final validation."
        )

    try:
        scratch_result_h5ad.unlink(missing_ok=True)
        scratch_input.unlink(missing_ok=True)
    except Exception as cleanup_error:
        print("[INFO] scratch cleanup skipped:", repr(cleanup_error))

    # Free memory.
    del adata
    gc.collect()

    print(f"[DONE] {strategy} elapsed_min={elapsed_min:.2f}")
    return {
        "strategy": strategy,
        "run_summary_tsv": str(run_summary_tsv),
        "cluster_summary_tsv": str(cluster_summary_tsv),
        "cell_scores_tsv": str(cell_scores_tsv),
        "cnv_leiden_summary_tsv": str(cnv_leiden_summary_tsv),
        "result_h5ad": str(result_h5ad),
    }


In [6]:

# =========================
# Run all selected strategies sequentially
# =========================

all_run_records = []

for _, row in manifest.iterrows():
    rec = run_one_infercnvpy_strategy(row)
    all_run_records.append(rec)

RUN_MANIFEST_TSV = METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv"
run_manifest = pd.DataFrame(all_run_records)
write_tsv_replace(run_manifest, RUN_MANIFEST_TSV)

display(run_manifest)



[START] core_conservative
Input: G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_inputs\GBM_core_4datasets_CNV_test_subset_core_conservative_HVG8000_r0_6_countsX.h5ad
[FOUND SCRATCH RESULT] E:\glioma-cnv-scratch\main_analysis_11\results\infercnvpy_core_conservative_HVG8000_r0_6_result.h5ad
[COMPATIBILITY REPAIR] created /obsm/cnv_umap as a hard-link to /obsm/X_cnv_umap
[RESULT VALIDATED] E:\glioma-cnv-scratch\main_analysis_11\results\infercnvpy_core_conservative_HVG8000_r0_6_result.h5ad | shape= (32640, 15163) | X_cnv= (32640, 1256) | obsm= ['X_cnv', 'X_cnv_pca', 'X_cnv_umap', 'X_scVI', 'X_umap', 'cnv_umap']
[RESUME] completed scratch infercnvpy result is valid; publishing without recomputation.
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_core_conservative_HVG8000_r0_6_run_summary.tsv shape=(1, 18)
[RECOVERED WITHOUT RECOMPUTATION] core_conservative

[START] core_plus_lymphoid
Input: G:\Repository\glioma-cnv-single-cell-atlas\data\inte

100%|██████████| 39/39 [00:26<00:00,  1.50it/s]


computing PCA
    with n_comps=30
    finished (0:00:02)
computing neighbors
    finished (0:01:16)
running Leiden clustering
    finished (0:00:31)
computing UMAP
    finished (0:00:35)
[COMPATIBILITY] aliased obsm['X_cnv_umap'] to obsm['cnv_umap']
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_core_plus_lymphoid_HVG8000_r0_6_cluster_summary.tsv shape=(8, 30)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_core_plus_lymphoid_HVG8000_r0_6_cnv_leiden_summary.tsv shape=(19, 18)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_core_plus_lymphoid_HVG8000_r0_6_cell_scores.tsv shape=(38807, 13)
[RESULT VALIDATED] E:\glioma-cnv-scratch\main_analysis_11\results\infercnvpy_core_plus_lymphoid_HVG8000_r0_6_result.h5ad | shape= (38807, 15163) | X_cnv= (38807, 1256) | obsm= ['X_cnv', 'X_cnv_pca', 'X_cnv_umap', 'X_scVI', 'X_umap', 'cnv_umap']
[RESULT VALIDATED] G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_

100%|██████████| 45/45 [00:30<00:00,  1.46it/s]


computing PCA
    with n_comps=30
    finished (0:00:02)
computing neighbors
    finished (0:00:05)
running Leiden clustering
    finished (0:00:31)
computing UMAP
    finished (0:00:41)
[COMPATIBILITY] aliased obsm['X_cnv_umap'] to obsm['cnv_umap']
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_extended_myeloid_HVG8000_r0_6_cluster_summary.tsv shape=(9, 30)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_extended_myeloid_HVG8000_r0_6_cnv_leiden_summary.tsv shape=(19, 18)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_extended_myeloid_HVG8000_r0_6_cell_scores.tsv shape=(45000, 13)
[RESULT VALIDATED] E:\glioma-cnv-scratch\main_analysis_11\results\infercnvpy_extended_myeloid_HVG8000_r0_6_result.h5ad | shape= (45000, 15163) | X_cnv= (45000, 1256) | obsm= ['X_cnv', 'X_cnv_pca', 'X_cnv_umap', 'X_scVI', 'X_umap', 'cnv_umap']
[RESULT VALIDATED] G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_results\

100%|██████████| 15/15 [00:11<00:00,  1.34it/s]


computing PCA
    with n_comps=30
    finished (0:00:01)
computing neighbors
    finished (0:00:03)
running Leiden clustering
    finished (0:00:02)
computing UMAP
    finished (0:00:12)
[COMPATIBILITY] aliased obsm['X_cnv_umap'] to obsm['cnv_umap']
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_review_cycling_HVG8000_r0_6_cluster_summary.tsv shape=(3, 30)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_review_cycling_HVG8000_r0_6_cnv_leiden_summary.tsv shape=(17, 18)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_review_cycling_HVG8000_r0_6_cell_scores.tsv shape=(14640, 13)
[RESULT VALIDATED] E:\glioma-cnv-scratch\main_analysis_11\results\infercnvpy_review_cycling_HVG8000_r0_6_result.h5ad | shape= (14640, 15163) | X_cnv= (14640, 1256) | obsm= ['X_cnv', 'X_cnv_pca', 'X_cnv_umap', 'X_scVI', 'X_umap', 'cnv_umap']
[RESULT VALIDATED] G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_results\infercnv

100%|██████████| 20/20 [00:13<00:00,  1.49it/s]


computing PCA
    with n_comps=30
    finished (0:00:01)
computing neighbors
    finished (0:00:03)
running Leiden clustering
    finished (0:00:08)
computing UMAP
    finished (0:00:16)
[COMPATIBILITY] aliased obsm['X_cnv_umap'] to obsm['cnv_umap']
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_review_ambiguous_HVG8000_r0_6_cluster_summary.tsv shape=(4, 30)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_review_ambiguous_HVG8000_r0_6_cnv_leiden_summary.tsv shape=(9, 18)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_review_ambiguous_HVG8000_r0_6_cell_scores.tsv shape=(19443, 13)
[RESULT VALIDATED] E:\glioma-cnv-scratch\main_analysis_11\results\infercnvpy_review_ambiguous_HVG8000_r0_6_result.h5ad | shape= (19443, 15163) | X_cnv= (19443, 1256) | obsm= ['X_cnv', 'X_cnv_pca', 'X_cnv_umap', 'X_scVI', 'X_umap', 'cnv_umap']
[RESULT VALIDATED] G:\Repository\glioma-cnv-single-cell-atlas\data\interim\integrated\CNV_results\i

,strategy,run_summary_tsv,cluster_summary_tsv,cell_scores_tsv,cnv_leiden_summary_tsv,result_h5ad
0,core_conservative,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
1,core_plus_lymphoid,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
2,extended_myeloid,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
3,review_cycling,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...
4,review_ambiguous,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\dat...


In [7]:

# =========================
# Combine strategy-level outputs
# =========================

run_manifest = pd.read_csv(METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv", sep="\t")

cluster_summaries = []
cell_score_summaries = []
run_summaries = []
cnv_leiden_summaries = []

for _, r in run_manifest.iterrows():
    p = Path(r["cluster_summary_tsv"])
    if p.exists():
        cluster_summaries.append(pd.read_csv(p, sep="\t"))
    p = Path(r["run_summary_tsv"])
    if p.exists():
        run_summaries.append(pd.read_csv(p, sep="\t"))
    p = Path(r["cnv_leiden_summary_tsv"])
    if p.exists():
        cnv_leiden_summaries.append(pd.read_csv(p, sep="\t"))

combined_cluster = pd.concat(cluster_summaries, ignore_index=True) if cluster_summaries else pd.DataFrame()
combined_run = pd.concat(run_summaries, ignore_index=True) if run_summaries else pd.DataFrame()
combined_cnv_leiden = pd.concat(cnv_leiden_summaries, ignore_index=True) if cnv_leiden_summaries else pd.DataFrame()

COMBINED_CLUSTER_TSV = METADATA_DIR / "infercnvpy_combined_cluster_summary_HVG8000_r0_6.tsv"
COMBINED_RUN_TSV = METADATA_DIR / "infercnvpy_combined_run_summary_HVG8000_r0_6.tsv"
COMBINED_CNV_LEIDEN_TSV = METADATA_DIR / "infercnvpy_combined_cnv_leiden_summary_HVG8000_r0_6.tsv"

write_tsv_replace(combined_cluster, COMBINED_CLUSTER_TSV)
write_tsv_replace(combined_run, COMBINED_RUN_TSV)
write_tsv_replace(combined_cnv_leiden, COMBINED_CNV_LEIDEN_TSV)

display(combined_run)
display(combined_cluster.head())


if combined_cluster.shape != (30, 30):
    raise AssertionError(
        f"Historical combined biological-cluster shape changed: {combined_cluster.shape}"
    )

if combined_run.shape != (5, 18):
    raise AssertionError(
        f"Historical combined run-summary shape changed: {combined_run.shape}"
    )

if combined_cnv_leiden.shape[1] != 18:
    raise AssertionError(
        f"CNV-Leiden summary column structure changed: {combined_cnv_leiden.shape}"
    )

print(
    "Historical combined CNV-Leiden shape: (91, 18) | current:",
    combined_cnv_leiden.shape,
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_combined_cluster_summary_HVG8000_r0_6.tsv shape=(30, 30)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_combined_run_summary_HVG8000_r0_6.tsv shape=(5, 18)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_combined_cnv_leiden_summary_HVG8000_r0_6.tsv shape=(85, 18)


,strategy,input_h5ad,result_h5ad,cell_scores_tsv,cluster_summary_tsv,cnv_leiden_summary_tsv,n_cells,n_genes,n_cnv_windows,preprocessing,elapsed_minutes,figures,high_threshold,moderate_threshold,reference_mad,reference_median,reference_q95,reference_q99
0,core_conservative,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,32640,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,NaN,NaN,0.025629,0.021525,0.002768,0.013318,0.020419,0.024693
1,core_plus_lymphoid,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,38807,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,3.398,G:\Repository\glioma-cnv-single-cell-atlas\fig...,0.025536,0.021259,0.002885,0.012704,0.020549,0.025131
2,extended_myeloid,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,45000,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,2.442,G:\Repository\glioma-cnv-single-cell-atlas\fig...,0.030094,0.024566,0.004394,0.010548,0.024566,0.029647
3,review_cycling,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,14640,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,0.727,G:\Repository\glioma-cnv-single-cell-atlas\fig...,0.025605,0.021505,0.002766,0.013304,0.020389,0.024693
4,review_ambiguous,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\dat...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,G:\Repository\glioma-cnv-single-cell-atlas\met...,19443,15163,1256,normalize_total_1e4_log1p_from_raw_counts_X,0.956,G:\Repository\glioma-cnv-single-cell-atlas\fig...,0.025773,0.021650,0.002781,0.013404,0.020553,0.024805


,strategy,cluster,n_cells,cnv_group_majority,cnv_group_majority_fraction,mean_cnv_abs_score,median_cnv_abs_score,q75_cnv_abs_score,q90_cnv_abs_score,reference_median,...,top_patient_id,top_patient_id_fraction,top_sample_id,top_sample_id_fraction,top_annotation_marker_based_HVG8000_r0_6,top_annotation_marker_based_HVG8000_r0_6_fraction,top_lineage_marker_based_HVG8000_r0_6,top_lineage_marker_based_HVG8000_r0_6_fraction,top_annotation_confidence_HVG8000_r0_6,top_annotation_confidence_HVG8000_r0_6_fraction
0,core_conservative,0,6387,query,1.0,0.016490,0.016386,0.019710,0.022933,0.013318,...,04,0.184907,GSM5518602_ndGBM-01-D,0.069203,putative_tumor_OPC_NPC_like,1.0,putative_tumor_like,1.0,moderate_high,1.0
1,core_conservative,11,5702,query,1.0,0.017953,0.017346,0.020821,0.024641,0.013318,...,GSM8546836_BT368,0.155559,GSM8546836_BT368,0.155559,putative_tumor_NPC_neuronal_like,1.0,putative_tumor_like,1.0,moderate,1.0
2,core_conservative,14,2640,reference,1.0,0.013675,0.013792,0.016613,0.019417,0.013318,...,02,0.241667,GSM5518634_ndGBM-05,0.133333,endothelial,1.0,vascular_endothelial,1.0,high,1.0
3,core_conservative,4,6432,query,1.0,0.013421,0.012362,0.016055,0.019929,0.013318,...,GSM8546833_BT356,0.158116,GSM8546833_BT356,0.158116,putative_tumor_hypoxia_stress,1.0,putative_tumor_like_stress,1.0,moderate_high,1.0
4,core_conservative,7,6479,query,1.0,0.016166,0.015603,0.019086,0.022568,0.013318,...,04,0.124865,GSM8546834_BT363,0.117302,putative_tumor_AC_like_reactive,1.0,putative_tumor_like,1.0,moderate_high,1.0


Historical combined CNV-Leiden shape: (91, 18) | current: (85, 18)


In [8]:

# =========================
# Consensus table across strategies
# =========================

if combined_cluster.empty:
    raise ValueError("combined_cluster is empty")

# Keep query/review clusters; references are used to estimate thresholds, not final malignant calls.
query_like = combined_cluster[combined_cluster["cnv_group_majority"].isin(["query", "review_query", "query_review", "query_putative_tumor_like"])].copy()

# In case the group values are exactly only reference/query in your inputs.
if query_like.empty:
    query_like = combined_cluster[combined_cluster["cnv_group_majority"].astype(str).str.contains("query", case=False, na=False)].copy()

if query_like.empty:
    print("[WARNING] No query-like clusters found by cnv_group_majority. Using all non-reference clusters.")
    query_like = combined_cluster[combined_cluster["cnv_group_majority"] != "reference"].copy()

call_strength = {
    "CNV_low_or_reference_like": 0,
    "CNV_intermediate_candidate": 1,
    "CNV_high_candidate": 2,
}
query_like["cnv_call_strength"] = query_like["cnv_call"].map(call_strength).fillna(0).astype(int)

cons_rows = []
for cl, df in query_like.groupby("cluster", observed=True):
    n_strat = df["strategy"].nunique()
    n_high = int((df["cnv_call"] == "CNV_high_candidate").sum())
    n_intermediate_or_high = int((df["cnv_call_strength"] >= 1).sum())
    strategies_high = ",".join(df.loc[df["cnv_call"] == "CNV_high_candidate", "strategy"].astype(str).tolist())
    strategies_intermediate_or_high = ",".join(df.loc[df["cnv_call_strength"] >= 1, "strategy"].astype(str).tolist())
    median_score_median = float(df["median_cnv_abs_score"].median())
    max_effect = float(df["robust_effect_vs_reference_median"].max())

    # Consensus logic.
    main_core = df[df["strategy"].isin(["core_conservative", "core_plus_lymphoid", "extended_myeloid"])]
    core_n = main_core.shape[0]
    core_high = int((main_core["cnv_call"] == "CNV_high_candidate").sum())
    core_int_high = int((main_core["cnv_call_strength"] >= 1).sum())

    if core_n > 0 and core_high >= 2:
        consensus = "robust_CNV_high_candidate"
    elif core_n > 0 and core_int_high >= 2:
        consensus = "recurrent_CNV_intermediate_candidate"
    elif n_high >= 1:
        consensus = "single_strategy_CNV_high_review"
    elif n_intermediate_or_high >= 1:
        consensus = "single_strategy_CNV_intermediate_review"
    else:
        consensus = "CNV_low_or_reference_like"

    # Carry forward annotation columns if present.
    row = {
        "cluster": str(cl),
        "n_strategies_observed": int(n_strat),
        "n_high_calls": n_high,
        "n_intermediate_or_high_calls": n_intermediate_or_high,
        "strategies_high": strategies_high,
        "strategies_intermediate_or_high": strategies_intermediate_or_high,
        "median_of_strategy_median_cnv_abs_score": median_score_median,
        "max_robust_effect_vs_reference_median": max_effect,
        "consensus_cnv_call": consensus,
    }
    for col in [
        "top_annotation_marker_based_HVG8000_r0_6",
        "top_lineage_marker_based_HVG8000_r0_6",
        "top_cnv_role_HVG8000_r0_6",
        "top_bias_status_HVG8000_r0_6",
    ]:
        if col in df.columns:
            vc = df[col].astype(str).value_counts(dropna=False)
            row[col] = vc.index[0] if len(vc) else "NA"
    cons_rows.append(row)

consensus = pd.DataFrame(cons_rows).sort_values([
    "consensus_cnv_call", "max_robust_effect_vs_reference_median"
], ascending=[True, False])

CONSENSUS_TSV = METADATA_DIR / "infercnvpy_consensus_cluster_calls_HVG8000_r0_6.tsv"
write_tsv_replace(consensus, CONSENSUS_TSV)

display(consensus)


if consensus.shape != (7, 11):
    raise AssertionError(
        f"Historical consensus structure changed: {consensus.shape}"
    )

_expected_query_clusters = {"0", "2", "4", "7", "9", "11", "12"}
_observed_query_clusters = set(consensus["cluster"].astype(str))

if _observed_query_clusters != _expected_query_clusters:
    raise RuntimeError(
        "Query/review cluster set differs from the archived run: "
        f"{sorted(_observed_query_clusters)}"
    )

_expected_strategy_coverage = {
    "0": 3,
    "2": 1,
    "4": 3,
    "7": 3,
    "9": 1,
    "11": 3,
    "12": 1,
}
_observed_strategy_coverage = dict(
    zip(
        consensus["cluster"].astype(str),
        consensus["n_strategies_observed"].astype(int),
    )
)

if _observed_strategy_coverage != _expected_strategy_coverage:
    raise RuntimeError(
        "Consensus strategy coverage differs from the archived run."
    )

if not (
    consensus["consensus_cnv_call"]
    == "CNV_low_or_reference_like"
).all():
    raise RuntimeError(
        "infercnvpy-only consensus differs from the archived article run. "
        "Stop before downstream CNV QC and inspect the current versus "
        "historical strategy results."
    )

print("[HISTORICAL INFERCNVPY CONSENSUS REPRODUCED]")


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_consensus_cluster_calls_HVG8000_r0_6.tsv shape=(7, 11)


,cluster,n_strategies_observed,n_high_calls,n_intermediate_or_high_calls,strategies_high,strategies_intermediate_or_high,median_of_strategy_median_cnv_abs_score,max_robust_effect_vs_reference_median,consensus_cnv_call,top_annotation_marker_based_HVG8000_r0_6,top_lineage_marker_based_HVG8000_r0_6
5,11,3,0,0,,,0.017346,1.182955,CNV_low_or_reference_like,putative_tumor_NPC_neuronal_like,putative_tumor_like
0,0,3,0,0,,,0.016515,0.995507,CNV_low_or_reference_like,putative_tumor_OPC_NPC_like,putative_tumor_like
3,7,3,0,0,,,0.015950,0.880782,CNV_low_or_reference_like,putative_tumor_AC_like_reactive,putative_tumor_like
1,2,1,0,0,,,0.016681,0.823502,CNV_low_or_reference_like,cycling_proliferating_cells,cycling_ambiguous
2,4,3,0,0,,,0.012362,0.295791,CNV_low_or_reference_like,putative_tumor_hypoxia_stress,putative_tumor_like_stress
6,12,1,0,0,,,0.013881,0.115639,CNV_low_or_reference_like,MES_ECM_stromal_like_ambiguous,mesenchymal_stromal_ambiguous
4,9,1,0,0,,,0.013401,-0.000631,CNV_low_or_reference_like,ambiguous_mixed_Hb_T_NK_HLA,ambiguous_mixed_or_doublet


[HISTORICAL INFERCNVPY CONSENSUS REPRODUCED]


In [9]:

# =========================
# Historical floating-point audit
# =========================

historical_thresholds = pd.DataFrame([
    ["core_conservative", 0.013315, 0.002776, 0.020502, 0.024749, 0.021548, 0.025664],
    ["core_plus_lymphoid", 0.012679, 0.002861, 0.020540, 0.025113, 0.021161, 0.025402],
    ["extended_myeloid", 0.010540, 0.004405, 0.024548, 0.029754, 0.024548, 0.030131],
    ["review_cycling", 0.013304, 0.002774, 0.020502, 0.024749, 0.021530, 0.025644],
    ["review_ambiguous", 0.013400, 0.002789, 0.020650, 0.024936, 0.021671, 0.025806],
], columns=[
    "strategy",
    "reference_median",
    "reference_mad",
    "reference_q95",
    "reference_q99",
    "moderate_threshold",
    "high_threshold",
])

numeric_cols = [
    "reference_median",
    "reference_mad",
    "reference_q95",
    "reference_q99",
    "moderate_threshold",
    "high_threshold",
]

current_thresholds = combined_run[["strategy", *numeric_cols]].copy()

threshold_audit = historical_thresholds.merge(
    current_thresholds,
    on="strategy",
    suffixes=("_historical", "_current"),
)

for column in numeric_cols:
    threshold_audit[f"{column}_abs_difference"] = np.abs(
        threshold_audit[f"{column}_current"]
        - threshold_audit[f"{column}_historical"]
    )

threshold_audit["historical_combined_cnv_leiden_rows"] = 91
threshold_audit["current_combined_cnv_leiden_rows"] = combined_cnv_leiden.shape[0]

THRESHOLD_AUDIT_TSV = (
    METADATA_DIR
    / "infercnvpy_historical_numeric_audit_HVG8000_r0_6.tsv"
)
write_tsv_replace(threshold_audit, THRESHOLD_AUDIT_TSV)

display(threshold_audit)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_historical_numeric_audit_HVG8000_r0_6.tsv shape=(5, 21)


,strategy,reference_median_historical,reference_mad_historical,reference_q95_historical,reference_q99_historical,moderate_threshold_historical,high_threshold_historical,reference_median_current,reference_mad_current,reference_q95_current,...,moderate_threshold_current,high_threshold_current,reference_median_abs_difference,reference_mad_abs_difference,reference_q95_abs_difference,reference_q99_abs_difference,moderate_threshold_abs_difference,high_threshold_abs_difference,historical_combined_cnv_leiden_rows,current_combined_cnv_leiden_rows
0,core_conservative,0.013315,0.002776,0.020502,0.024749,0.021548,0.025664,0.013318,0.002768,0.020419,...,0.021525,0.025629,2.703575e-06,0.000008,0.000083,0.000056,0.000023,0.000035,91,85
1,core_plus_lymphoid,0.012679,0.002861,0.020540,0.025113,0.021161,0.025402,0.012704,0.002885,0.020549,...,0.021259,0.025536,2.538347e-05,0.000024,0.000009,0.000018,0.000098,0.000134,91,85
2,extended_myeloid,0.010540,0.004405,0.024548,0.029754,0.024548,0.030131,0.010548,0.004394,0.024566,...,0.024566,0.030094,8.448280e-06,0.000011,0.000018,0.000107,0.000018,0.000037,91,85
3,review_cycling,0.013304,0.002774,0.020502,0.024749,0.021530,0.025644,0.013304,0.002766,0.020389,...,0.021505,0.025605,4.162435e-07,0.000008,0.000113,0.000056,0.000025,0.000039,91,85
4,review_ambiguous,0.013400,0.002789,0.020650,0.024936,0.021671,0.025806,0.013404,0.002781,0.020553,...,0.021650,0.025773,3.714580e-06,0.000008,0.000097,0.000131,0.000021,0.000033,91,85


In [10]:

# =========================
# Output manifest
# =========================

items = {
    "readiness_report": METADATA_DIR / "infercnvpy_run_readiness_HVG8000_r0_6.tsv",
    "strategy_run_manifest": METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv",
    "combined_run_summary": METADATA_DIR / "infercnvpy_combined_run_summary_HVG8000_r0_6.tsv",
    "combined_cluster_summary": METADATA_DIR / "infercnvpy_combined_cluster_summary_HVG8000_r0_6.tsv",
    "combined_cnv_leiden_summary": METADATA_DIR / "infercnvpy_combined_cnv_leiden_summary_HVG8000_r0_6.tsv",
    "consensus_cluster_calls": METADATA_DIR / "infercnvpy_consensus_cluster_calls_HVG8000_r0_6.tsv",
}

# Add per-strategy outputs from run_manifest.
if (METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv").exists():
    rm = pd.read_csv(METADATA_DIR / "infercnvpy_strategy_run_manifest_HVG8000_r0_6.tsv", sep="\t")
    for _, r in rm.iterrows():
        strategy = r["strategy"]
        for col in ["run_summary_tsv", "cluster_summary_tsv", "cell_scores_tsv", "cnv_leiden_summary_tsv", "result_h5ad"]:
            if col in rm.columns:
                items[f"{strategy}_{col}"] = Path(r[col])

rows = []
for item, path in items.items():
    path = Path(path)
    rows.append({
        "item": item,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan,
    })

out_manifest = pd.DataFrame(rows)
OUT_MANIFEST_TSV = METADATA_DIR / "infercnvpy_output_manifest_HVG8000_r0_6.tsv"
write_tsv_replace(out_manifest, OUT_MANIFEST_TSV)

display(out_manifest)


if out_manifest.shape != (31, 4):
    raise AssertionError(
        f"Historical output-manifest structure changed: {out_manifest.shape}"
    )

if not out_manifest["exists"].all():
    missing = out_manifest.loc[
        ~out_manifest["exists"],
        ["item", "path"],
    ]
    raise RuntimeError(
        "Missing infercnvpy outputs:\n"
        + missing.to_string(index=False)
    )

print("\nDONE — canonical infercnvpy strategy analysis completed.")
print("Run manifest:", run_manifest.shape)
print("Combined run summary:", combined_run.shape)
print("Combined biological-cluster summary:", combined_cluster.shape)
print("Combined CNV-Leiden summary:", combined_cnv_leiden.shape)
print("Consensus:", consensus.shape)
print(
    "All consensus calls:",
    sorted(consensus["consensus_cnv_call"].astype(str).unique().tolist()),
)
print("Output manifest:", out_manifest.shape)
print("Scratch root:", SCRATCH_ROOT)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\infercnvpy_output_manifest_HVG8000_r0_6.tsv shape=(31, 4)


,item,path,exists,size_mb
0,readiness_report,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
1,strategy_run_manifest,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.003
2,combined_run_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.008
3,combined_cluster_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.013
4,combined_cnv_leiden_summary,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.020
5,consensus_cluster_calls,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
6,core_conservative_run_summary_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
7,core_conservative_cluster_summary_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.003
8,core_conservative_cell_scores_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,6.732
9,core_conservative_cnv_leiden_summary_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.005



DONE — canonical infercnvpy strategy analysis completed.
Run manifest: (5, 6)
Combined run summary: (5, 18)
Combined biological-cluster summary: (30, 30)
Combined CNV-Leiden summary: (85, 18)
Consensus: (7, 11)
All consensus calls: ['CNV_low_or_reference_like']
Output manifest: (31, 4)
Scratch root: E:\glioma-cnv-scratch\main_analysis_11


## Expected successful completion

```text
Run manifest:                         (5, 6)
Combined run summary:                 (5, 18)
Combined biological-cluster summary: (30, 30)
Consensus:                            (7, 11)
Output manifest:                      (31, 4)

All consensus calls:
['CNV_low_or_reference_like']
```

Each result H5AD must contain 15,163 genes, `X_cnv` with 1,256 windows,
`cnv_abs_mean_score`, `cnv_leiden`, and `cnv_umap`.

The archived diagnostic CNV-Leiden combined table had shape `(91, 18)`.
The current row count is reported rather than used as an article-defining hard
checkpoint because notebook-10's canonical sampled cell identities can differ
from the old pre-renaming sampled objects.

If execution is interrupted, rerun from the beginning. Completed strategies are
validated and skipped automatically.
